In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn imbalanced-learn seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss
)

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
dataset = load_dataset('google-research-datasets/go_emotions')

In [ ]:
dataset

In [ ]:
dataset['train'][0]

In [ ]:
label_names = dataset["train"].features["labels"].feature.names

print("Number of emotions:", len(label_names))

for i, label in enumerate(label_names):
    print(i, "→", label)

In [ ]:
id2label = {
    i: label
    for i, label in enumerate(label_names)
}

label2id = {
    label: i
    for i, label in enumerate(label_names)
}

In [ ]:
print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))

In [ ]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

In [ ]:
train_df.head()

In [ ]:
print("Train missing values:")
print(train_df.isnull().sum())

print("\nValidation missing values:")
print(val_df.isnull().sum())

print("\nTest missing values:")
print(test_df.isnull().sum())

In [ ]:
print("Empty texts:", (train_df["text"].str.strip() == "").sum())

In [ ]:
print("Duplicate texts in train:",
      train_df["text"].duplicated().sum())

print("Duplicate texts in validation:",
      val_df["text"].duplicated().sum())

print("Duplicate texts in test:",
      test_df["text"].duplicated().sum())

In [ ]:
train_texts = set(train_df["text"])
val_texts = set(val_df["text"])
test_texts = set(test_df["text"])

print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test:", len(train_texts & test_texts))
print("Validation ∩ Test:", len(val_texts & test_texts))

In [ ]:
train_df["num_labels"] = train_df["labels"].apply(len)

print(train_df["num_labels"].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    x=train_df["num_labels"]
)

plt.xlabel("Number of emotions per text")
plt.ylabel("Number of samples")
plt.title("Multi-label Distribution in GoEmotions")

plt.show()

In [ ]:
emotion_counts = Counter()

for labels in train_df["labels"]:
    for label_id in labels:
        emotion_counts[label_id] += 1

In [ ]:
emotion_frequency = pd.DataFrame({
    "emotion_id": list(emotion_counts.keys()),
    "emotion": [
        id2label[i]
        for i in emotion_counts.keys()
    ],
    "count": list(emotion_counts.values())
})

emotion_frequency = emotion_frequency.sort_values(
    "count",
    ascending=False
)

emotion_frequency

In [ ]:
plt.figure(figsize=(12, 7))

sns.barplot(
    data=emotion_frequency,
    x="count",
    y="emotion"
)

plt.xlabel("Number of samples")
plt.ylabel("Emotion")
plt.title("GoEmotions Emotion Distribution")

plt.show()

In [ ]:
max_count = emotion_frequency["count"].max()
min_count = emotion_frequency["count"].min()

imbalance_ratio = max_count / min_count

print("Maximum class count:", max_count)
print("Minimum class count:", min_count)
print("Imbalance ratio:", round(imbalance_ratio, 2))

In [ ]:
train_df["text_length"] = train_df["text"].str.len()

train_df["word_count"] = train_df["text"].str.split().str.len()

In [ ]:
print(train_df["word_count"].describe())

In [ ]:
plt.figure(figsize=(10, 5))

sns.histplot(
    train_df["word_count"],
    bins=50
)

plt.xlabel("Number of words")
plt.ylabel("Frequency")
plt.title("GoEmotions Text Length Distribution")

plt.show()

In [ ]:
multi_label_examples = train_df[
    train_df["labels"].apply(len) > 1
]

for _, row in multi_label_examples.head(10).iterrows():

    emotions = [
        id2label[label_id]
        for label_id in row["labels"]
    ]

    print("TEXT:", row["text"])
    print("EMOTIONS:", emotions)
    print("-" * 80)

In [ ]:
NUM_LABELS = len(label_names)

def create_multihot(labels):
    target = np.zeros(NUM_LABELS, dtype=np.float32)

    for label in labels:
        target[label] = 1.0

    return target

In [ ]:
example_labels = train_df.iloc[0]["labels"]

print("Original:", example_labels)
print("Multi-hot:", create_multihot(example_labels))

In [ ]:
train_df["multi_hot"] = train_df["labels"].apply(create_multihot)
val_df["multi_hot"] = val_df["labels"].apply(create_multihot)
test_df["multi_hot"] = test_df["labels"].apply(create_multihot)

In [ ]:
import re
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from collections import Counter

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss
)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
def clean_text(text):

    text = str(text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
train_df["clean_text"] = train_df["text"].apply(clean_text)
val_df["clean_text"] = val_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

In [ ]:
def tokenize(text):
    return text.split()

In [ ]:
text = train_df.iloc[0]["clean_text"]

print(text)
print(tokenize(text))

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_IDX = 0
UNK_IDX = 1

MAX_VOCAB_SIZE = 30000

counter = Counter()

for text in train_df["clean_text"]:
    counter.update(tokenize(text))


In [ ]:
vocab = {
    PAD_TOKEN: PAD_IDX,
    UNK_TOKEN: UNK_IDX
}

for word, count in counter.most_common(MAX_VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

In [ ]:
def text_to_ids(text):

    tokens = tokenize(text)

    return [
        vocab.get(token, UNK_IDX)
        for token in tokens
    ]

In [ ]:
example = train_df.iloc[0]["clean_text"]

print("Text:")
print(example)

print("\nToken IDs:")
print(text_to_ids(example))

In [ ]:
MAX_LENGTH = 64

In [ ]:
class GoEmotionsDataset(Dataset):

    def __init__(self, dataframe, max_length=64):

        self.texts = dataframe["clean_text"].tolist()
        self.labels = dataframe["multi_hot"].tolist()

        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]
        label = self.labels[idx]

        token_ids = text_to_ids(text)

        # Truncate
        token_ids = token_ids[:self.max_length]

        # Padding
        padding_length = self.max_length - len(token_ids)

        token_ids += [PAD_IDX] * padding_length

        return {
            "input_ids": torch.tensor(
                token_ids,
                dtype=torch.long
            ),

            "labels": torch.tensor(
                label,
                dtype=torch.float
            )
        }

In [ ]:
train_dataset = GoEmotionsDataset(
    train_df,
    MAX_LENGTH
)

val_dataset = GoEmotionsDataset(
    val_df,
    MAX_LENGTH
)

test_dataset = GoEmotionsDataset(
    test_df,
    MAX_LENGTH
)

In [ ]:
sample = train_dataset[0]

print("Input shape:", sample["input_ids"].shape)
print("Label shape:", sample["labels"].shape)

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

In [ ]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)

In [ ]:
class BiLSTMEmotionModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=200,
        hidden_dim=128,
        num_layers=2,
        dropout=0.3,
        num_classes=28
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, input_ids):

        x = self.embedding(input_ids)

        output, (hidden, cell) = self.lstm(x)

        # Last forward hidden state
        forward_hidden = hidden[-2]

        # Last backward hidden state
        backward_hidden = hidden[-1]

        # Combine
        representation = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1
        )

        representation = self.dropout(
            representation
        )

        logits = self.classifier(
            representation
        )

        return logits

In [ ]:
model = BiLSTMEmotionModel(
    vocab_size=len(vocab),
    embedding_dim=200,
    hidden_dim=128,
    num_layers=2,
    dropout=0.3,
    num_classes=NUM_LABELS
)

model = model.to(device)

print(model)

In [ ]:
criterion = nn.BCEWithLogitsLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-3,
    weight_decay=1e-4
)

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0

    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def evaluate(
    model,
    loader,
    criterion,
    device,
    threshold=0.5
):

    model.eval()

    total_loss = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids)

            loss = criterion(
                logits,
                labels
            )

            total_loss += loss.item()

            probabilities = torch.sigmoid(
                logits
            )

            predictions = (
                probabilities >= threshold
            ).int()

            all_labels.append(
                labels.cpu().numpy()
            )

            all_predictions.append(
                predictions.cpu().numpy()
            )

    all_labels = np.vstack(all_labels)
    all_predictions = np.vstack(all_predictions)

    metrics = {

        "loss": total_loss / len(loader),

        "micro_f1": f1_score(
            all_labels,
            all_predictions,
            average="micro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            all_labels,
            all_predictions,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            all_labels,
            all_predictions,
            average="weighted",
            zero_division=0
        ),

        "precision_micro": precision_score(
            all_labels,
            all_predictions,
            average="micro",
            zero_division=0
        ),

        "recall_micro": recall_score(
            all_labels,
            all_predictions,
            average="micro",
            zero_division=0
        ),

        "hamming_loss": hamming_loss(
            all_labels,
            all_predictions
        )
    }

    return metrics

In [ ]:
EPOCHS = 10

history = []

best_val_f1 = 0

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_metrics = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        **val_metrics
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Micro F1: {val_metrics['micro_f1']:.4f} | "
        f"Macro F1: {val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_val_f1:

        best_val_f1 = val_metrics["macro_f1"]

        torch.save(
            model.state_dict(),
            "best_bilstm_goemotions.pt"
        )

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1/10 | Train Loss: 0.1464 | Val Loss: 0.1233 | Micro F1: 0.2672 | Macro F1: 0.0948


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 2/10 | Train Loss: 0.1179 | Val Loss: 0.1114 | Micro F1: 0.4465 | Macro F1: 0.1790


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Epoch 8/10 | Train Loss: 0.0525 | Val Loss: 0.1391 | Micro F1: 0.4853 | Macro F1: 0.3161


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 9/10 | Train Loss: 0.0467 | Val Loss: 0.1427 | Micro F1: 0.4726 | Macro F1: 0.3263


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 10/10 | Train Loss: 0.0416 | Val Loss: 0.1502 | Micro F1: 0.4690 | Macro F1: 0.3307


In [ ]:
EPOCHS = 10

history = []

best_val_f1 = 0

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_metrics = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        **val_metrics
    })

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Micro F1: {val_metrics['micro_f1']:.4f} | "
        f"Macro F1: {val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_val_f1:

        best_val_f1 = val_metrics["macro_f1"]

        torch.save(
            model.state_dict(),
            "best_bilstm_goemotions.pt"
        )

In [ ]:
history_df = pd.DataFrame(history)

history_df

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BiLSTM Training")

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_df["epoch"],
    history_df["micro_f1"],
    label="Micro F1"
)

plt.plot(
    history_df["epoch"],
    history_df["macro_f1"],
    label="Macro F1"
)

plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("BiLSTM Validation Performance")

plt.legend()
plt.show()

In [ ]:
model.load_state_dict(
    torch.load(
        "best_bilstm_goemotions.pt",
        map_location=device
    )
)

In [ ]:
test_metrics = evaluate(
    model,
    test_loader,
    criterion,
    device
)

test_metrics